# String Extraction & Replacement with RegEx

Filtering text is only half the battle. Often, you need to **extract** a specific part of a text string into a new column, or **replace** messy patterns dynamically.

In this module, you will learn:
* How to extract substrings using Capture Groups `()` and `.str.extract()`.
* How to find and replace text matching complex patterns using `.str.replace()` with RegEx.
* The difference between Series `.str.replace()` and DataFrame `.replace()`.

### Simple Explanation & Real-World Analogy

* **Extraction** is like a sorting machine at a recycling plant. The machine looks at a mixed stream of garbage, locates only the aluminum cans, pulls them out, and places them into a clean container of their own.
* **Replacement** is like a text editor's "Find and Replace" tool, but instead of finding one exact word, it can swap out any sequence of characters that matches a rule (like removing all exclamation marks and numbers from a word list).

### Code Examples

Let's work with a DataFrame containing messy descriptions of product dimensions.

In [1]:
import pandas as pd

data = {
    'Item': ['Item_A', 'Item_B', 'Item_C'],
    'Raw_Specs': ['Size is 15cm (Height)', 'Weight is 120g (Heavy)', 'Size is 45cm (Large)']
}
df = pd.DataFrame(data)
print(df)

     Item               Raw_Specs
0  Item_A   Size is 15cm (Height)
1  Item_B  Weight is 120g (Heavy)
2  Item_C    Size is 45cm (Large)


#### Extracting Numbers using `.str.extract()`
If we want to pull out the size numbers (like `15` and `45`) from `Raw_Specs`, we use `.str.extract()`.
* We write a RegEx pattern containing a **Capture Group** `()`. Whatever matches inside the parentheses will be extracted.
* `(\d+)` tells Pandas: "Find digits, capture them, and pull them out."


In [2]:
# Extract the numeric digits from the specs column
df['Size_Value'] = df['Raw_Specs'].str.extract(r'(\d+)')
print(df)

     Item               Raw_Specs Size_Value
0  Item_A   Size is 15cm (Height)         15
1  Item_B  Weight is 120g (Heavy)        120
2  Item_C    Size is 45cm (Large)         45


*(Note: Since Item_B had "120g", it extracted "120". We can clean this further next.)*


#### Dynamic Replacement using `.str.replace()`
Let's clean the `Raw_Specs` column by removing any parentheses and the text inside them (e.g., removing `(Height)`, `(Heavy)`, `(Large)`).
* In RegEx, `\s*` matches optional spaces.
* `\(` and `\)` match literal open and close parentheses.
* `.*?` matches any characters inside.

In [3]:
# Replace the parenthesis sections with nothing (an empty string)
# Note: In Pandas 1.2+, we must explicitly set regex=True to avoid warning messages
df['Clean_Specs'] = df['Raw_Specs'].str.replace(r'\s*\(.*?\)', '', regex=True)
print(df[['Raw_Specs', 'Clean_Specs']])

                Raw_Specs     Clean_Specs
0   Size is 15cm (Height)    Size is 15cm
1  Weight is 120g (Heavy)  Weight is 120g
2    Size is 45cm (Large)    Size is 45cm



#### Series `.str.replace()` vs DataFrame `.replace()`
* **`Series.str.replace()`** is designed for partial matches and regex replacements inside a single text Series. It always outputs string types.
* **`DataFrame.replace()`** is used to replace *exact, entire values* across multiple columns at once (e.g., changing all occurrences of exactly `None` or `?` to `NaN`). It can handle numeric and categorical values as well.

### Common Mistakes Beginners Make
1. **Forgetting Capture Groups in `.str.extract()`:**
   If you write `df['col'].str.extract(r'\d+')` without parentheses, Pandas will raise a `ValueError: pattern contains no capture groups`. You must wrap what you want to extract in parentheses: `r'(\d+)'`.
2. **Incorrect escaping of RegEx symbols:**
   Parentheses `()`, brackets `[]`, and dots `.` have special meanings in RegEx. If you want to replace a literal parenthesized word, you must use double backslashes or a raw string prefix `r'\s*\(.*?\)'`. Otherwise, RegEx will interpret the parentheses as a capture group rather than literal characters.


#### Exercise 1 (Hard)
You have a DataFrame of transactional log strings:
```python
logs = pd.DataFrame({
    'ID': [1, 2, 3],
    'Info': ['User Alice purchased code #A102', 'User Bob purchased code #B205', 'User Charlie purchased code #C909']
})
```
Write Pandas code to:
1. Extract the usernames (`Alice`, `Bob`, `Charlie`) into a new column called `'Username'`. (Hint: Look for words starting with a capital letter right after `'User '`).
2. Extract the item code (e.g., `A102`, `B205`, `C909`) into a new column called `'Product_Code'`. (Hint: Look for characters following the `'#'` symbol).



In [4]:
import pandas as pd

logs = pd.DataFrame({
    'ID': [1, 2, 3],
    'Info': ['User Alice purchased code #A102', 'User Bob purchased code #B205', 'User Charlie purchased code #C909']
})

# Extract username: Looks for words right after "User "
logs['Username'] = logs['Info'].str.extract(r'User\s+([A-Za-z]+)')

# Extract product code: Looks for alphanumeric digits right after "#"
logs['Product_Code'] = logs['Info'].str.extract(r'#([A-Za-z0-9]+)')

print(logs[['Username', 'Product_Code']])

  Username Product_Code
0    Alice         A102
1      Bob         B205
2  Charlie         C909
